# Invasive bird data collection

## 1. Load dependencies

In [3]:
#for missing packages use pip install <package_name>

import pandas as pd
from pathlib import Path
import requests
import time  
from tqdm import tqdm 

## 2. Fetch bird data present in sinas x globi merge

In [ ]:
repo_root         = Path.cwd().parent # get repo root

# Define input files
sinas_file = repo_root / 'data' / 'sinas_matched_species.csv'
globi_net_file = repo_root / 'data' / 'matched_globi_network.csv'
globi_high_file = repo_root / 'data' / 'matched_globi_network_higher_order.csv'

# Define output files
sinas_output = 'isolated_birds_sinas.csv'
globi_net_output = 'isolated_birds_globi_network.csv'
globi_high_output = 'isolated_birds_globi_higher_order.csv'

print("Step 1: Loading datasets...")
df_sinas = pd.read_csv(sinas_file)
df_globi_net = pd.read_csv(globi_net_file)
df_globi_high = pd.read_csv(globi_high_file)

print("Step 2: Isolating bird records from SINAS data...")
# GBIF Class ID 212 corresponds to Class: Aves (Birds)
birds_sinas = df_sinas[df_sinas['gbif_class_id'] == 212.0]

# Retain all original information and save to a new file
birds_sinas.to_csv(sinas_output, index=False)
print(f" -> Saved complete bird species data to: {sinas_output}")

print("Step 3: Extracting bird lineage taxonomic keys...")
# Gather all valid bird keys (Species, Genus, Family, Order, Class) 
# to capture higher-rank mappings in the GLOBI files
bird_keys = set()
rank_cols = ['gbif_id', 'gbif_genus_id', 'gbif_family_id', 'gbif_order_id', 'gbif_class_id']
for col in rank_cols:
    if col in birds_sinas.columns:
        bird_keys.update(birds_sinas[col].dropna().unique())

# Standardize keys to floats for cross-dataset type matching
bird_keys = {float(k) for k in bird_keys}

# Helper function to check if a value belongs to a bird lineage
def is_bird_id(val):
    try:
        return float(val) in bird_keys
    except (ValueError, TypeError):
        return False

print("Step 4: Isolating bird records from GLOBI network datasets...")
# Filter both standard and higher-order networks
birds_globi_net = df_globi_net[
    df_globi_net['source_gbif_id'].apply(is_bird_id) | 
    df_globi_net['target_gbif_id'].apply(is_bird_id)
]

birds_globi_high = df_globi_high[
    df_globi_high['source_gbif_id'].apply(is_bird_id) | 
    df_globi_high['target_gbif_id'].apply(is_bird_id)
]

# Save isolated networks
birds_globi_net.to_csv(globi_net_output, index=False)
birds_globi_high.to_csv(globi_high_output, index=False)
print(f" -> Saved isolated standard GLOBI network to: {globi_net_output}")
print(f" -> Saved isolated higher-order GLOBI network to: {globi_high_output}")

print("Step 5: Verifying total unique bird species...")
# Count unique bird species IDs from SINAS
unique_species_ids = set(birds_sinas['gbif_id'].dropna().unique())

# Double check if any species-level bird IDs exist in the filtered GLOBI network 
# that might not be captured in the SINAS slice
for _, row in birds_globi_net.iterrows():
    if row['source_rank'] == 'SPECIES' and is_bird_id(row['source_gbif_id']):
        unique_species_ids.add(float(row['source_gbif_id']))
    if row['target_rank'] == 'SPECIES' and is_bird_id(row['target_gbif_id']):
        unique_species_ids.add(float(row['target_gbif_id']))

print("\n" + "="*50)
print("              DATA ISOLATION SUMMARY              ")
print("="*50)
print(f"Total Unique Bird Species Present: {len(unique_species_ids)}")
print(f"SINAS Bird Rows Retained:          {len(birds_sinas)}")
print(f"GLOBI Network Interactions:         {len(birds_globi_net)}")
print(f"GLOBI Higher-Order Interactions:    {len(birds_globi_high)}")
print("="*50)

Step 1: Loading datasets...
Step 2: Isolating bird records from SINAS data...
 -> Saved complete bird species data to: isolated_birds_sinas.csv
Step 3: Extracting bird lineage taxonomic keys...
Step 4: Isolating bird records from GLOBI network datasets...
 -> Saved isolated standard GLOBI network to: isolated_birds_globi_network.csv
 -> Saved isolated higher-order GLOBI network to: isolated_birds_globi_higher_order.csv
Step 5: Verifying total unique bird species...

              DATA ISOLATION SUMMARY              
Total Unique Bird Species Present: 1138
SINAS Bird Rows Retained:          29395
GLOBI Network Interactions:         68752
GLOBI Higher-Order Interactions:    24921


## 3. Verify number of GBIF occurrences for associated species

In [ ]:
# 1. Load only the isolated SINAS bird dataset
sinas_file = 'isolated_birds_sinas.csv'

print("Loading isolated SINAS bird dataset...")
df_sinas = pd.read_csv(sinas_file)

# 2. Compile unique species GBIF IDs strictly from the SINAS file
print("Compiling unique species GBIF IDs from SINAS...")
unique_bird_ids = set(df_sinas['gbif_id'].dropna().unique())

# Clean up IDs
unique_bird_ids = [int(float(uid)) for uid in unique_bird_ids if pd.notna(uid)]
total_species = len(unique_bird_ids)
print(f" -> Found {total_species} unique bird species IDs to query.")

# 3. Rate-limited execution loop with Retry Logic
total_observations = 0
species_count_mapping = {}
DELAY_SECONDS = 0.5  # Base delay between successful calls
MAX_RETRIES = 3      # Number of times to retry a timed-out ID

print(f"\nQuerying GBIF API sequentially (with automated retries for timeouts)...")

progress_bar = tqdm(unique_bird_ids, desc="Fetching GBIF counts", unit="species")

for uid in progress_bar:
    url = f"https://api.gbif.org/v1/occurrence/search?taxonKey={uid}&limit=0"
    
    count = 0
    success = False
    
    # Retry loop for handling transient network drops
    for attempt in range(MAX_RETRIES):
        try:
            # Increased timeout to 15 seconds to give slow server responses a buffer
            response = requests.get(url, timeout=15)
            
            if response.status_code == 200:
                count = response.json().get('count', 0)
                success = True
                break  # Success! Break out of the retry loop
            elif response.status_code == 429:
                # If GBIF explicitly throttles us, back off aggressively
                tqdm.write(f" [Rate Limit] Status 429 for ID {uid}. Cooling down...")
                time.sleep(5)
            else:
                tqdm.write(f" [Warning] API status {response.status_code} for ID {uid}")
                break  # Non-200 framework error; retrying won't help
                
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            # Calculate a progressive back-off wait time (1s, 3s, 5s...)
            wait_time = 1 + (attempt * 2)
            tqdm.write(f" [Timeout/Connection Error] ID {uid} failed (Attempt {attempt + 1}/{MAX_RETRIES}). Retrying in {wait_time}s...")
            time.sleep(wait_time)
            
        except Exception as e:
            tqdm.write(f" [Unexpected Error] ID {uid}: {e}")
            break

    # Record whatever data we got (or 0 if all retries failed)
    species_count_mapping[uid] = count
    total_observations += count
    
    # Update progress bar metadata
    progress_bar.set_postfix(running_obs=f"{total_observations:,}")
    
    # Standard polite rate-limiting delay between individual species cycles
    time.sleep(DELAY_SECONDS)

print("\n" + "="*50)
print("           SINAS GBIF OBSERVATION REPORT          ")
print("="*50)
print(f"Total Unique Bird Species Checked: {total_species}")
print(f"Total Combined Observations:       {total_observations:,}")
print("="*50)

# 4. Export results
output_csv = 'sinas_birds_gbif_counts.csv'
print(f"\nExporting individual counts to '{output_csv}'...")
df_counts = pd.DataFrame(list(species_count_mapping.items()), columns=['gbif_id', 'gbif_observation_count'])
df_counts.to_csv(output_csv, index=False)
print("Done!")

Loading isolated SINAS bird dataset...
Compiling unique species GBIF IDs from SINAS...
 -> Found 1138 unique bird species IDs to query.

Querying GBIF API sequentially...


Fetching GBIF counts:  90%|████████▉ | 1022/1138 [22:26<01:57,  1.02s/species, running_obs=1,093,093,005]

 [Error] Request failed for ID 2498390: HTTPSConnectionPool(host='api.gbif.org', port=443): Read timed out. (read timeout=10)


Fetching GBIF counts: 100%|██████████| 1138/1138 [24:35<00:00,  1.30s/species, running_obs=1,182,851,156]


           SINAS GBIF OBSERVATION REPORT          
Total Unique Bird Species Checked: 1138
Total Combined Observations:       1,182,851,156

Exporting individual counts to 'sinas_birds_gbif_counts.csv'...
Done!
